**IMPORTS**

In [1]:
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DeiTFeatureExtractor, ViTModel
from PIL import Image
from tqdm import tqdm

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**CONFIG**

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = "../checkpoints/early_fusion_stylometric/"
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5

CODEBERT_CKPT = "../checkpoints/codebert_only/checkpoint-2322/"
VIT_CKPT = "../checkpoints/vit_only/deit_epoch_3.pt"

TEXT_DIR = "../Text_Files/Train"
IMAGE_DIR = "../snapshots/Train"
TEST_BASE = "../snapshots"

**LOAD MODELS**

In [3]:
from transformers import ViTForImageClassification, AutoImageProcessor

print("Loading fine-tuned CodeBERT...")
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
text_model = AutoModel.from_pretrained(CODEBERT_CKPT).to(DEVICE)
text_model.eval()

Loading fine-tuned CodeBERT...


Some weights of RobertaModel were not initialized from the model checkpoint at ../checkpoints/codebert_only/checkpoint-2322/ and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dr

In [4]:
print("Loading fine-tuned DeiT model...")
image_processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

# Load your trained classification model first
trained_model = ViTForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    ignore_mismatched_sizes=True
)

trained_model.load_state_dict(torch.load(VIT_CKPT, map_location=DEVICE))
trained_model.to(DEVICE)

# Extract just the ViT base model for embeddings
vit_model = trained_model.vit# This gives you the pure ViTModel
vit_model.to(DEVICE)
vit_model.eval()

Loading fine-tuned DeiT model...


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 8630.26it/s]
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.
Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 7584.64it/s]
Some weights of ViTForImageClassification were not initialized from the model checkpoint at facebook/deit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=768, out_features=3072, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
          (d

**DATASET**

In [5]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset
from PIL import Image

# Stylometric feature extractor
def extract_stylometric_features(code: str) -> np.ndarray:
    lines = code.splitlines()
    avg_line_length = np.mean([len(line) for line in lines]) if lines else 0
    num_lines = len(lines)
    num_tokens = len(code.split())
    num_chars = len(code)
    return np.array([avg_line_length, num_lines, num_tokens, num_chars], dtype=np.float32)


class FusionDataset(Dataset):
    def __init__(self, text_dir, image_dir, tokenizer, image_processor):
        self.text_paths = []
        self.image_paths = []
        self.labels = []
        self.tokenizer = tokenizer
        self.image_processor = image_processor

        # Scan text folders
        for label_folder in sorted(os.listdir(text_dir)):
            label_path = os.path.join(text_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            label = int(label_folder.split("_")[1])  # e.g., "Label_0" -> 0
            for txt_file in sorted(os.listdir(label_path)):
                if txt_file.endswith(".txt"):
                    self.text_paths.append(os.path.join(label_path, txt_file))
                    self.labels.append(label)

        # Scan image folders
        self.image_paths = []
        for label_folder in sorted(os.listdir(image_dir)):
            label_path = os.path.join(image_dir, label_folder)
            if not os.path.isdir(label_path):
                continue
            for img_file in sorted(os.listdir(label_path)):
                if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.image_paths.append(os.path.join(label_path, img_file))

        assert len(self.text_paths) == len(self.image_paths), "Text and image counts must match!"

    def __len__(self):
        return len(self.text_paths)

    def __getitem__(self, idx):
        # ----- TEXT -----
        with open(self.text_paths[idx], "r") as f:
            text = f.read()

        # Tokenized text
        encoding = self.tokenizer(
            text, return_tensors="pt", truncation=True, padding="max_length", max_length=512
        )
        input_ids = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        # Stylometric features
        style_feats = torch.tensor(extract_stylometric_features(text), dtype=torch.float32)

        # ----- IMAGE -----
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image_tensor = self.image_processor(images=image, return_tensors="pt")
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].squeeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return input_ids, attention_mask, image_tensor, style_feats, label


**DATA LOADING**

In [6]:
dataset = FusionDataset(TEXT_DIR, IMAGE_DIR, tokenizer, image_processor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

**MODEL ARCHITECTURE**

In [7]:
import torch
import torch.nn as nn

class FusionClassifier(nn.Module):
    def __init__(self, text_model, vit_model, hidden_dim=512, num_classes=2):
        super().__init__()
        self.text_model = text_model
        self.vit_model = vit_model

        # freeze backbone models
        for p in self.text_model.parameters():
            p.requires_grad = False
        for p in self.vit_model.parameters():
            p.requires_grad = False

        text_emb_dim = text_model.config.hidden_size
        vit_emb_dim = vit_model.config.hidden_size
        style_dim = 4  # from extract_stylometric_features()

        self.classifier = nn.Sequential(
            nn.Linear(text_emb_dim + vit_emb_dim + style_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, input_ids, attention_mask, image_tensor, style_feats):
        # Text embedding (CLS token)
        text_outputs = self.text_model(input_ids=input_ids, attention_mask=attention_mask)
        text_cls = text_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Image embedding (CLS token)
        image_outputs = self.vit_model(**{k: v for k, v in image_tensor.items()})
        image_cls = image_outputs.last_hidden_state[:, 0, :]  # [batch, hidden]

        # Concatenate text, image, and stylometric features
        fused = torch.cat([text_cls, image_cls, style_feats], dim=1)

        logits = self.classifier(fused)
        return logits


**OPTIMIZERS && LOSS**

In [8]:
model = FusionClassifier(text_model, vit_model).to(DEVICE)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

**TRAINING LOOP && CHECKPOINTING**

In [ ]:
import os
import torch
from tqdm import tqdm

# ---------------- Configuration ----------------
START_EPOCH = 0
CHECKPOINT_DIR = OUTPUT_DIR  # same as where you save models
EPOCHS = 14
# ------------------------------------------------

# 🔹 Find the last saved checkpoint (if any)
checkpoint_files = [
    f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("fusion_epoch_") and f.endswith(".pt")
]
if checkpoint_files:
    # Sort by epoch number
    checkpoint_files.sort(key=lambda x: int(x.split("_")[-1].split(".")[0]))
    latest_ckpt = checkpoint_files[-1]
    START_EPOCH = int(latest_ckpt.split("_")[-1].split(".")[0])
    ckpt_path = os.path.join(CHECKPOINT_DIR, latest_ckpt)
    print(f"🔄 Loading checkpoint from: {ckpt_path}")
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
else:
    print("⚠️ No checkpoint found. Starting training from scratch.")

# ---------------- Training Loop ----------------
for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids, attention_mask, image_tensor, style_feats, labels = batch

        # Move to device
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        style_feats = style_feats.to(DEVICE)
        labels = labels.to(DEVICE)
        for k in image_tensor:
            image_tensor[k] = image_tensor[k].to(DEVICE)

        # Forward pass
        logits = model(input_ids, attention_mask, image_tensor, style_feats)
        loss = criterion(logits, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Avg Loss: {avg_loss:.4f}")

    # Save checkpoint with updated epoch
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"fusion_epoch_{epoch+1}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"✅ Checkpoint saved: {ckpt_path}")


🔄 Loading checkpoint from: ../checkpoints/early_fusion_stylometric/fusion_epoch_10.pt


Epoch 11/12: 100%|██████████| 774/774 [01:26<00:00,  8.96it/s]


Epoch 11/12 | Avg Loss: 0.2545
✅ Checkpoint saved: ../checkpoints/early_fusion_stylometric/fusion_epoch_11.pt


Epoch 12/12: 100%|██████████| 774/774 [01:27<00:00,  8.89it/s]


Epoch 12/12 | Avg Loss: 0.2380
✅ Checkpoint saved: ../checkpoints/early_fusion_stylometric/fusion_epoch_12.pt


**TESTING**

In [9]:
from torch.utils.data import DataLoader
from tqdm import tqdm

def test_on_dataset(text_dir, image_dir, tokenizer, image_processor, fusion_model, batch_size=4):
    """🧪 Test the fusion model (text + image + stylometric) on a single dataset"""

    dataset = FusionDataset(text_dir, image_dir, tokenizer, image_processor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    fusion_model.eval()
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc="Testing Dataset 📝"):
            # Unpack batch
            input_ids, attention_mask, image_tensor, style_feats, labels = batch

            # Move tensors to device
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            style_feats = style_feats.to(DEVICE)
            labels = labels.to(DEVICE)
            for k in image_tensor:
                image_tensor[k] = image_tensor[k].to(DEVICE)

            # Forward pass
            logits = fusion_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_tensor=image_tensor,
                style_feats=style_feats
            )

            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    accuracy = total_correct / total_samples
    print(f"✅ Dataset Accuracy: {accuracy * 100:.2f}% 🎉")
    return accuracy


In [14]:
FUSION_CKPT = "../checkpoints/early_fusion_stylometric/fusion_epoch_12.pt"
fusion_model = FusionClassifier(text_model, vit_model)
fusion_model.to(DEVICE)
# Load checkpoint
state_dict = torch.load(FUSION_CKPT, map_location=DEVICE)
fusion_model.load_state_dict(state_dict)
fusion_model.eval()

FusionClassifier(
  (text_model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNo

**RESULTS ON 3 EPOCH CKPT**

In [29]:
for i in range(10):
    print(f"Test_{i}")
    test_on_dataset(f"../Text_Files/Test_{i}", f"../snapshots/Test_{i}", tokenizer, image_processor, fusion_model, batch_size=4)

Test_0


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.04it/s]


✅ Dataset Accuracy: 78.74% 🎉
Test_1


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.24it/s]


✅ Dataset Accuracy: 76.55% 🎉
Test_2


...

✅ Dataset Accuracy: 69.46% 🎉
Test_8


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.87it/s]


✅ Dataset Accuracy: 72.55% 🎉
Test_9


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.10it/s]

✅ Dataset Accuracy: 69.46% 🎉


**RESULT ON 10 EPOCH CKPT**

In [11]:
for i in range(10):
    print(f"Test_{i}")
    test_on_dataset(f"../Text_Files/Test_{i}", f"../snapshots/Test_{i}", tokenizer, image_processor, fusion_model, batch_size=4)

Test_0


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.78it/s]


✅ Dataset Accuracy: 79.44% 🎉
Test_1


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.16it/s]


✅ Dataset Accuracy: 77.45% 🎉
Test_2


...

✅ Dataset Accuracy: 71.06% 🎉
Test_8


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.57it/s]


✅ Dataset Accuracy: 73.75% 🎉
Test_9


Testing Dataset 📝: 100%|██████████| 251/251 [00:13<00:00, 18.00it/s]

✅ Dataset Accuracy: 71.06% 🎉


**RESULT ON 12 EPOCH CKPT**

In [15]:
for i in range(10):
    print(f"Test_{i}")
    test_on_dataset(f"../Text_Files/Test_{i}", f"../snapshots/Test_{i}", tokenizer, image_processor, fusion_model, batch_size=4)

Test_0


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.53it/s]


✅ Dataset Accuracy: 79.54% 🎉
Test_1


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.64it/s]


✅ Dataset Accuracy: 77.54% 🎉
Test_2


...

✅ Dataset Accuracy: 71.76% 🎉
Test_8


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.63it/s]


✅ Dataset Accuracy: 73.55% 🎉
Test_9


Testing Dataset 📝: 100%|██████████| 251/251 [00:14<00:00, 17.72it/s]

✅ Dataset Accuracy: 71.76% 🎉
